<h1>02 · Exploración y Preparación de Features <a id="inicio"> </h1>
<em>Notebook desarrollado por Evelyn Cabrera Arias.</em><br><br>

<b>Repositorio:</b> <code>credit-risk-score</code><br>
<b>Dataset:</b> <code>dataset_bank_customer.csv</code><br>
<b>Objetivo:</b> Ingeniería de variables, encoding, escalamiento y división train/test.

<hr>

<h3 style="color: #0d47a1;">Estructura:</h3>

<a href="#seccion-1"  style="color:black; text-decoration:underline; display:block; margin:3px 0; font-size:15px;">1. Setup</a>
<a href="#seccion-2"  style="color:black; text-decoration:underline; display:block; margin:3px 0; font-size:15px;">2. Ingeniería de features</a>
<a href="#seccion-3"  style="color:black; text-decoration:underline; display:block; margin:3px 0; font-size:15px;">3. Encoding de variables categóricas</a>
<a href="#seccion-4"  style="color:black; text-decoration:underline; display:block; margin:3px 0; font-size:15px;">4. Selección de features para el modelo</a>
<a href="#seccion-5"  style="color:black; text-decoration:underline; display:block; margin:3px 0; font-size:15px;">5. Escalamiento y split train/test</a>
<a href="#seccion-6"  style="color:black; text-decoration:underline; display:block; margin:3px 0; font-size:15px;">6. Guardar datasets procesados</a>

## 1. Setup <a id="seccion-1"></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/raw/dataset_bank_customer.csv', sep=';')
print(f"Shape original: {df.shape}")
df.head()

Shape original: (10000, 16)


,id_cliente,score_originacion,pais,sexo,edad,tenencia,saldo_promedio,numero_productos,tarjeta_credito,miembro_activo,ingresos_mensuales,fecha_apertura,estado_civil,segmento_cliente,deuda_total,moroso
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,8/12/2010,Soltero,C,0.00,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,1/07/2016,Soltero,C,167615.72,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,3/02/2016,Divorciado,D,319321.60,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,23/04/2018,Soltero,B,0.00,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,18/02/2012,Soltero,A,251021.64,0


## 2. Ingeniería de features <a id="seccion-2"></a>

In [2]:
# 2.1 Convertir fecha_apertura y calcular antigüedad en meses
df['fecha_apertura'] = pd.to_datetime(df['fecha_apertura'], dayfirst=True, errors='coerce')
fecha_ref = pd.Timestamp('2024-01-01')
df['antiguedad_meses'] = ((fecha_ref - df['fecha_apertura']) / pd.Timedelta(days=30.44)).round(0).astype(int)

# 2.2 Ratio deuda/ingreso
df['ratio_deuda_ingreso'] = np.where(
    df['ingresos_mensuales'] > 0,
    (df['deuda_total'] / (df['ingresos_mensuales'] * 12)).round(4),
    0
)

# 2.3 Ratio utilización de saldo
df['ratio_saldo_ingreso'] = np.where(
    df['ingresos_mensuales'] > 0,
    (df['saldo_promedio'] / (df['ingresos_mensuales'] * 12)).round(4),
    0
)

print("Features creadas: antiguedad_meses, ratio_deuda_ingreso, ratio_saldo_ingreso")
df[['id_cliente','antiguedad_meses','ratio_deuda_ingreso','ratio_saldo_ingreso']].head()

Features creadas: antiguedad_meses, ratio_deuda_ingreso, ratio_saldo_ingreso


,id_cliente,antiguedad_meses,ratio_deuda_ingreso,ratio_saldo_ingreso
0,15634602,157,0.0000,0.0000
1,15647311,90,0.1241,0.0621
2,15619304,95,0.2336,0.1168
3,15701354,68,0.0000,0.0000
4,15737888,142,0.2645,0.1323


## 3. Encoding de variables categóricas <a id="seccion-3"></a>

In [3]:
# 3.1 Label Encoding para sexo (binaria)
le_sexo = LabelEncoder()
df['sexo_encoded'] = le_sexo.fit_transform(df['sexo'])
print("sexo:", dict(zip(le_sexo.classes_, le_sexo.transform(le_sexo.classes_))))

# 3.2 One-Hot Encoding para estado_civil y segmento_cliente
df = pd.get_dummies(df, columns=['estado_civil', 'segmento_cliente'], drop_first=True)

# 3.3 One-Hot para pais
df = pd.get_dummies(df, columns=['pais'], drop_first=True)

print(f"\nShape después del encoding: {df.shape}")
print("Columnas nuevas:", [c for c in df.columns if '_' in c and c not in ['id_cliente','fecha_apertura']])

sexo: {'Female': np.int64(0), 'Male': np.int64(1)}

Shape después del encoding: (10000, 25)
Columnas nuevas: ['score_originacion', 'saldo_promedio', 'numero_productos', 'tarjeta_credito', 'miembro_activo', 'ingresos_mensuales', 'deuda_total', 'antiguedad_meses', 'ratio_deuda_ingreso', 'ratio_saldo_ingreso', 'sexo_encoded', 'estado_civil_Divorciado', 'estado_civil_Soltero', 'estado_civil_Viudo', 'segmento_cliente_B', 'segmento_cliente_C', 'segmento_cliente_D', 'pais_Germany', 'pais_Spain']


## 4. Selección de features para el modelo <a id="seccion-4"></a>

In [4]:
features_numericas = [
    'edad', 'ingresos_mensuales', 'saldo_promedio', 'score_originacion',
    'antiguedad_meses', 'ratio_deuda_ingreso', 'ratio_saldo_ingreso', 'deuda_total'
]

features_categoricas = [c for c in df.columns if c.startswith(('estado_civil_','segmento_cliente_','pais_'))]
features_binarias = ['tarjeta_credito', 'miembro_activo', 'sexo_encoded', 'numero_productos']

features = features_numericas + features_categoricas + features_binarias
target   = 'moroso'

X = df[features].copy()
y = df[target]

print(f"Features seleccionadas: {len(features)}")
print(f"Distribución objetivo: {y.value_counts().to_dict()}")

Features seleccionadas: 20
Distribución objetivo: {0: 7963, 1: 2037}


## 5. Escalamiento y split train/test <a id="seccion-5"></a>

In [5]:
from sklearn.model_selection import train_test_split

# 5.1 Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5.2 Escalar solo numéricas continuas
scaler = StandardScaler()
X_train[features_numericas] = scaler.fit_transform(X_train[features_numericas])
X_test[features_numericas]  = scaler.transform(X_test[features_numericas])

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Distribución y_train:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (8000, 20) | Test: (2000, 20)
Distribución y_train:
moroso
0    0.796
1    0.204
Name: proportion, dtype: float64


## 6. Guardar datasets procesados <a id="seccion-6"></a>

In [6]:
import joblib

# Guardar datos procesados
X_train.to_csv('../data/clean/X_train.csv', index=False)
X_test.to_csv('../data/clean/X_test.csv', index=False)
y_train.to_csv('../data/clean/y_train.csv', index=False)
y_test.to_csv('../data/clean/y_test.csv', index=False)

# Guardar scaler y encoder para producción
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le_sexo, '../models/le_sexo.pkl')
joblib.dump(features, '../models/features_list.pkl')

print("✅ Datos limpios y artefactos guardados en /data/clean/ y /models/")

✅ Datos limpios y artefactos guardados en /data/clean/ y /models/


<h3 style="color: #0d47a1;">Siguiente paso: </h3>
`03_modelo.ipynb`
<em><a href="#inicio"  style="color:black; text-decoration:underline; display:block; margin:3px 0; font-size:15px;">Volver a inicio de 02_exploracion</a></em>